In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/schema.json
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00135.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00218.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00150.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00021.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00075.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00036.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/enwiki_namespace_0_00167.parquet
/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/

In [13]:
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc
import spacy

# Load spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

# Point at the dataset folder
DATASET_PATH = "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data"
dataset = ds.dataset(DATASET_PATH, format="parquet")

# Load only name + url for fast searching
print("Loading article index (this takes 2-3 min, only once)...")
name_table = dataset.to_table(columns=["name", "url"])
names_df = name_table.to_pandas()
print(f"Loaded {len(names_df):,} articles")

# Fast search function
def search_names(query, limit=10):
    matches = names_df[names_df['name'].str.contains(query, case=False, na=False, regex=False)]
    return matches.head(limit)['name'].tolist()

Loading article index (this takes 2-3 min, only once)...
Loaded 7,597,149 articles


In [14]:
def collect_links(parts):
    """Recursively walk sections/subsections and pull out every link's text."""
    found = []
    for part in parts:
        if not isinstance(part, dict):
            continue
        for link in part.get('links', []) or []:
            text = link.get('text') if isinstance(link, dict) else None
            if text:
                found.append(text)
        if 'has_parts' in part:
            found.extend(collect_links(part['has_parts']))
    return found


def get_related_info(row):
    """Return dict of {People: [...], Organizations: [...], Places: [...], Topics: [...]}"""
    sections = row.get('sections')
    
    # Normalize sections to a list of dicts
    if isinstance(sections, str):
        sections = json.loads(sections)
    elif isinstance(sections, np.ndarray):
        sections = sections.tolist()
    elif sections is None:
        sections = []
    elif not isinstance(sections, list):
        sections = []
    
    all_links = []
    for section in sections:
        if isinstance(section, dict):
            all_links.extend(collect_links(section.get('has_parts', [])))
    
    links = sorted(set(all_links))
    
    people, orgs, places, topics = [], [], [], []
    for phrase in links:
        doc = nlp(phrase)
        if not doc.ents:
            topics.append(phrase)
            continue
        label = doc.ents[0].label_
        if label == "PERSON":
            people.append(phrase)
        elif label == "ORG":
            orgs.append(phrase)
        elif label in ("GPE", "LOC"):
            places.append(phrase)
        else:
            topics.append(phrase)
    
    return {
        "People": people,
        "Organizations": orgs,
        "Places": places,
        "Topics": topics,
    }


def get_article_by_name(title):
    """Fetch full article row by exact name match."""
    table = dataset.to_table(
        filter=pc.equal(pc.field("name"), title)
    )
    if table.num_rows == 0:
        return None
    return table.to_pandas().iloc[0]

In [ ]:
# Demo — shows the pipeline working without input()
demo_matches = search_names("Indian Space", limit=5)
print("Matches:", demo_matches)

if demo_matches:
    row = get_article_by_name(demo_matches[0])
    if row is not None:
        print(f"\nArticle: {row['name']}")
        if row.get('abstract'):
            print(f"Abstract: {str(row['abstract'])[:200]}...")
        related = get_related_info(row)
        for cat, items in related.items():
            if items:
                print(f"\n{cat}: {items[:5]}")

In [ ]:
import gradio as gr

def gradio_search(query):
    if not query.strip():
        return "Please enter a search term.", gr.update(choices=[], value=None)
    matches = search_names(query, limit=10)
    if not matches:
        return f"No articles found for '{query}'", gr.update(choices=[], value=None)
    return f"Found {len(matches)} matches.", gr.update(choices=matches, value=matches[0])


def gradio_explore(article_name):
    if not article_name:
        return "No article selected.", ""
    
    row = get_article_by_name(article_name)
    if row is None:
        return f"Article '{article_name}' not found.", ""
    
    output = f"# 📖 {row['name']}\n\n"
    abstract = row.get('abstract', '')
    if abstract:
        output += f"**Abstract:** {abstract[:500]}...\n\n"
    
    related = get_related_info(row)
    for category, items in related.items():
        if items:
            output += f"### 📁 {category}\n"
            for item in items[:15]:
                output += f"- {item}\n"
            output += "\n"
    
    return output, ""  # second output reserved for future


with gr.Blocks(title="WikiKnowledge Explorer") as app:
    gr.Markdown("# 🔍 WikiKnowledge Explorer")
    gr.Markdown("Search a Wikipedia article and explore related information.")
    
    with gr.Row():
        search_input = gr.Textbox(label="Search", placeholder="e.g., Indian Space Research Organisation")
        search_btn = gr.Button("Search")
    
    status = gr.Textbox(label="Status", interactive=False)
    article_dropdown = gr.Dropdown(label="Select Article", choices=[])
    
    output_md = gr.Markdown(label="Related Information")
    
    search_btn.click(gradio_search, inputs=search_input, outputs=[status, article_dropdown])
    article_dropdown.change(gradio_explore, inputs=article_dropdown, outputs=[output_md, status])

app.launch(share=True)